### RAG Pipelines : Data Ingestion to Vector DB Pipeline

In [1]:
import os
from os.path import exists

from langchain_community.document_loaders import PyMuPDFLoader , PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

from numba.core.types import none


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4208\1197991040.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader , PyPDFLoader
D:\RAG project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # find all the PDFs recursively
    pdf_files = list(pdf_dir.rglob("*.pdf"))  # safer than os.listdir

    print(f"Found {len(pdf_files)} files")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['filetype'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Example usage
all_pdf_documents = process_all_pdfs("../data")


Found 5 files
Processing bert.pdf
Loaded 16 pages from bert.pdf
Processing dl.pdf
Loaded 1 pages from dl.pdf
Processing rag.pdf
Loaded 21 pages from rag.pdf
Processing transfromer.pdf
Loaded 11 pages from transfromer.pdf
Processing w2v.pdf
Loaded 12 pages from w2v.pdf

Total documents loaded: 61


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-04-29T17:36:03+00:00', 'author': 'Jacob Devlin ; Ming-Wei Chang ; Kenton Lee ; Kristina Toutanova', 'keywords': '', 'moddate': '2019-04-29T17:36:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'subject': 'N19-1 2019', 'title': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding', 'trapped': '/False', 'source': '..\\data\\pdf\\bert.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'bert.pdf', 'filetype': 'pdf'}, page_content='Proceedings of NAACL-HLT 2019, pages 4171–4186\nMinneapolis, Minnesota, June 2 - June 7, 2019. c⃝2019 Association for Computational Linguistics\n4171\nBERT: Pre-training of Deep Bidirectional Transformers for\nLanguage Understanding\nJacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova\nGoogle AI Language\n{jacobdevlin,mingweicha

In [4]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {str(split_docs[0].metadata)[:200]}...")

    return split_docs


In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 61 documents into 325 chunks

Example chunk:
Content: Proceedings of NAACL-HLT 2019, pages 4171–4186
Minneapolis, Minnesota, June 2 - June 7, 2019. c⃝2019 Association for Computational Linguistics
4171
BERT: Pre-training of Deep Bidirectional Transformer...
Metadata: {'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-04-29T17:36:03+00:00', 'author': 'Jacob Devlin ; Ming-Wei Chang ; Kenton Lee ; Kristina Toutanova', 'keyw...


[Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-04-29T17:36:03+00:00', 'author': 'Jacob Devlin ; Ming-Wei Chang ; Kenton Lee ; Kristina Toutanova', 'keywords': '', 'moddate': '2019-04-29T17:36:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'subject': 'N19-1 2019', 'title': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding', 'trapped': '/False', 'source': '..\\data\\pdf\\bert.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'bert.pdf', 'filetype': 'pdf'}, page_content='Proceedings of NAACL-HLT 2019, pages 4171–4186\nMinneapolis, Minnesota, June 2 - June 7, 2019. c⃝2019 Association for Computational Linguistics\n4171\nBERT: Pre-training of Deep Bidirectional Transformers for\nLanguage Understanding\nJacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova\nGoogle AI Language\n{jacobdevlin,mingweicha

### Embeddings and Vector Store DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Tuple,Any,Dict
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class Embedding_Manager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """model_name -> HuggingFace model name for sentence embeddings"""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the embedding model from HuggingFace SentenceTransformers"""
        try:
            print(f"Loading embedding model: {self.model_name} ...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.Embedding Dimension : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e

    def generate_embeddings(self, texts: List[str]):
        """
        Generate embeddings for a single string or a list of strings.
        Returns a numpy array.
        """
        if self.model is None:
            raise ValueError("Embedding model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}")
        return embeddings

embedding_manager = Embedding_Manager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3857.33it/s]


Model loaded successfully.Embedding Dimension : 384


### Vector Store

In [12]:
class Vector_Store:

    def __init__(self,collection_name:str = "pdf_documents", persist_directory : str = "../data/vector_store" ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:

            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
                )
            print(f"Vector store initialized successfully.Collection : {self.collection_name}")
            print(f"Existing documents in the collection : {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing collection {self.collection_name}: {e}")
            raise e

    def add_documents(self,documents : List[Any] , embeddings : np.ndarray):

        """Add documents to the collection
        Args:
        documents: List of documents to add
        embeddings: Embeddings to add
        """
        if len(documents) != len(embeddings):
            raise ValueError("Length of embeddings should be equal to length of documents")

        print(f"Adding {len(documents)} documents to Vector Store...")

        #Prepare Data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i , (doc,embedding)  in enumerate(zip(documents,embeddings)):
            #Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document Content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding)

            #Add to collection
            try:
                self.collection.add(
                    ids = ids,
                    embeddings = embeddings_list,
                    metadatas = metadatas,
                    documents = documents_text,
                )
                print(f"Successfully added document {len(documents)} to Vector Store")
                print(f"Total documents in collection: {self.collection.count()}")

            except Exception as e:
                print(f"Error adding document {len(documents)} to Vector Store: {e}")
                raise e

vector_store = Vector_Store()
vector_store





Vector store initialized successfully.Collection : pdf_documents
Existing documents in the collection : 0


In [14]:
#Convert text to embeddings
texts = [doc.page_content for doc in chunks]

#Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#Store into the Vector DB
vector_store.add_documents(chunks, embeddings)


Generating embeddings for 325 texts...


Batches: 100%|██████████| 11/11 [00:09<00:00,  1.16it/s]


Generated embeddings with shape (325, 384)
Adding 325 documents to Vector Store...
Successfully added document 325 to Vector Store
Total documents in collection: 1
Successfully added document 325 to Vector Store
Total documents in collection: 2
Successfully added document 325 to Vector Store
Total documents in collection: 3
Successfully added document 325 to Vector Store
Total documents in collection: 4
Successfully added document 325 to Vector Store
Total documents in collection: 5
Successfully added document 325 to Vector Store
Total documents in collection: 6
Successfully added document 325 to Vector Store
Total documents in collection: 7
Successfully added document 325 to Vector Store
Total documents in collection: 8
Successfully added document 325 to Vector Store
Total documents in collection: 9
Successfully added document 325 to Vector Store
Total documents in collection: 10
Successfully added document 325 to Vector Store
Total documents in collection: 11
Successfully added docum

### Retriever Pipeline from VectorStore

In [15]:
class RAGRetriever:
    """Handles query based Retrieval from vector store"""

    def __init__(self,vector_store: Vector_Store,embedding_manager: Embedding_Manager):
        """
        Initialize RAG Retriever
        Args:
        vector_store: Vector Store containing document embeddings
        embedding_manager:Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self , query:str , top_k : int = 5,score_threshold :float = 0.0) -> List[Dict[str, Any]]:

        """Retrieve documents for a query
        Args:
            query: Query to retrieve documents for
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: {query}")
        print(f"Top K : {top_k} , Score threshold : {score_threshold}:")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    """Convert distance to similarity score(ChromaDB uses cosine distance)"""

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents(after filtering).")

            else:
                print(f"No documents found for query: {query}")

            return retrieved_docs

        except Exception as e:
            print(f"Error retrieving documents for query: {query}")
            return []

rag_retriever = RAGRetriever(vector_store, embedding_manager)



In [16]:
rag_retriever

In [19]:
rag_retriever.retrieve("Recurrent Neural Net Language Model (RNNLM)")
# We get an output called context and this is fed to the LLM along with the prompt to generate output

Retrieving documents for query: Recurrent Neural Net Language Model (RNNLM)
Top K : 5 , Score threshold : 0.0:
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.54it/s]

Generated embeddings with shape (1, 384)
Retrieved 5 documents(after filtering).


[{'id': 'doc_1307c54b_284',
  'content': 'would require log2(V ) outputs to be evaluated, the Huffman tree based hierarchical softmax requires\nonly about log2(U nigram perplexity(V )). For example when the vocabulary size is one million\nwords, this results in about two times speedup in evaluation. While this is not crucial speedup for\nneural network LMs as the computational bottleneck is in theN ×D ×H term, we will later propose\narchitectures that do not have hidden layers and thus depend heavily on the efﬁciency of the softmax\nnormalization.\n2.2 Recurrent Neural Net Language Model (RNNLM)\nRecurrent neural network based language model has been proposed to overcome certain limitations\nof the feedforward NNLM, such as the need to specify the context length (the order of the modelN),\nand because theoretically RNNs can efﬁciently represent more complex patterns than the shallow\nneural networks [15, 2]. The RNN model does not have a projection layer; only input, hidden and',
  'me